# 适用于任意主题的 AI 测验 / 抽认卡生成器

## 练习目标（理念）

输入任意学习主题，用 LLM（OpenAI 云端或本地 Ollama 兼容接口）生成一组**测验题**或**抽认卡**，方便自学。

- **输入**：主题、题目数量、模式（`quiz` / `flashcard`）
- **输出**：按固定格式排好的问答或正反面卡片文本
- **额外要求**：无 API Key 时自动回退到本地 `llama3.2:latest`

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| `.env` / `OPENAI_API_KEY` | `load_dotenv` + `os.getenv` |
| Chat Completions | `client.chat.completions.create(...)` |
| 本地兼容端点 | `base_url=http://localhost:11434/v1` |
| 简单 UI | `ipywidgets` 收集输入并触发生成 |

## 怎么跑

1. 有云端密钥则写入 `.env` 的 `OPENAI_API_KEY`；否则先启动 Ollama 并拉取 `llama3.2:latest`
2. 运行导入与客户端单元格，再显示控件
3. 填写 Topic / 数量 / Mode，点生成按钮


In [8]:
# ========== 导入 + 选后端（OpenAI 云或本地 Ollama）+ 生成函数 ==========

# 导入标准库 os：读取环境变量里的 API Key
import os
# 从 openai 导入 OpenAI 客户端：云端与 Ollama 兼容端都用它
from openai import OpenAI
# 从 dotenv 导入 load_dotenv：把 .env 读进环境变量
from dotenv import load_dotenv

# 加载环境变量（若用 .env 存 API 密钥）；override=True 覆盖已有同名变量
load_dotenv(override=True)

# 从环境读取 OpenAI API Key；没有则为 None，后面走本地 Ollama
api_key = os.getenv('OPENAI_API_KEY')
# 本地 Ollama 的 OpenAI 兼容根地址（/v1）
ollama_url = "http://localhost:11434/v1"

# 打印当前读到的 Key（调试用；注意别把真实密钥提交到公开仓库）
print (f"API Key: {api_key}")

if not api_key:
    # 无云端密钥：api_key 占位为 "ollama"，base_url 指向本机
    client = OpenAI(api_key="ollama", base_url=ollama_url)
    # 本地模型名；需事先 ollama pull
    model="llama3.2:latest"
else:
    # 有密钥：走默认 OpenAI 云端地址，用 gpt-3.5-turbo
    model="gpt-3.5-turbo"
    client = OpenAI(api_key=api_key)

# 辅助函数：按主题生成测验题或抽认卡
def generate_quiz(topic, num_questions=5, mode="quiz"):
    """
    Generate quiz questions or flashcards for a given topic using OpenAI.
    mode: 'quiz' for Q&A, 'flashcard' for term/definition pairs
    """
    if mode == "quiz":
        # quiz 模式：要求 Q1/A1 这种成对格式（英文 prompt 不翻译）
        prompt = f"""
Generate {num_questions} quiz questions and answers about the topic: '{topic}'.
Format as:
Q1: ...\nA1: ...\nQ2: ...\nA2: ...
"""
    else:
        # flashcard 模式：正反面 Front/Back
        prompt = f"""
Generate {num_questions} flashcards about the topic: '{topic}'.
Format as:
Front 1: ...\nBack 1: ...\nFront 2: ...\nBack 2: ...
"""
    # 调用 Chat Completions：单条 user message；限制长度与一点随机性
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=512,
        temperature=0.7
    )
    # 取助手文本并去掉首尾空白
    return response.choices[0].message.content.strip()


API Key: None


In [5]:
# ========== UI：用 ipywidgets 收集主题、题量、模式 ==========

# 导入 ipywidgets：在笔记本里画交互控件
import ipywidgets as widgets
# 导入 display、Markdown：刷新输出区时展示结果
from IPython.display import display, Markdown

# 文本框：用户输入学习主题
topic_widget = widgets.Text(
    value='',
    placeholder='e.g. The Solar System',
    description='Topic:',
    disabled=False
)

# 滑块：题目/卡片数量，范围 1–20，默认 5
num_questions_widget = widgets.IntSlider(
    value=5,
    min=1,
    max=20,
    step=1,
    description='Questions:',
    disabled=False
)

# 切换按钮：quiz（测验）或 flashcard（抽认卡）；value 传给 generate_quiz 的 mode
mode_widget = widgets.ToggleButtons(
    options=[('Quiz', 'quiz'), ('Flashcard', 'flashcard')],
    value='quiz',
    description='Mode:',
    disabled=False,
    button_style=''
)

# 把三个控件显示在笔记本里
display(topic_widget, num_questions_widget, mode_widget)


Text(value='', description='Topic:', placeholder='e.g. The Solar System')

IntSlider(value=5, description='Questions:', max=20, min=1)

ToggleButtons(description='Mode:', options=(('Quiz', 'quiz'), ('Flashcard', 'flashcard')), value='quiz')

In [ ]:
# ========== 生成按钮：点击后调用 generate_quiz 并展示结果 ==========

# 成功样式按钮；文案保持原样（UI 字符串）
generate_button = widgets.Button(description="Generate Quiz/Flashcards", button_style='success')
# Output 控件：把生成过程与结果隔离在可清空的区域
output = widgets.Output()

def on_generate_clicked(b):
    # 每次点击先清空旧输出，避免结果堆叠
    output.clear_output()
    # 读控件当前值；主题去首尾空白
    topic = topic_widget.value.strip()
    num_questions = num_questions_widget.value
    mode = mode_widget.value
    if not topic:
        # 未填主题：提示用户（文案保持英文）
        with output:
            display(Markdown("**Please enter a topic.**"))
        return
    with output:
        # 先显示「正在生成」状态行
        display(Markdown(f"### Generating {num_questions} {'quiz questions' if mode=='quiz' else 'flashcards'} for: **{topic}** ..."))
        try:
            # 调上面的 generate_quiz；把返回文本放进代码块样式 Markdown
            result = generate_quiz(topic, num_questions, mode)
            display(Markdown(f"```\n{result}\n```"))
        except Exception as e:
            # 网络/模型错误时显示异常信息，不让整格崩溃
            display(Markdown(f"**Error:** {e}"))

# 把回调绑到按钮的 click 事件
generate_button.on_click(on_generate_clicked)
# 显示按钮与输出区
display(generate_button, output)


Button(button_style='success', description='Generate Quiz/Flashcards', style=ButtonStyle())

Output()